In [1]:
def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {col: 0 for col, dtype in df.dtypes if dtype in ["int", "bigint", "double"]}
    df = df.fillna(fill_dict).fillna(fill_dict_numeric)

    return df

StatementMeta(, 42b8ced0-3399-483a-977c-e778d114b0d4, 3, Finished, Available, Finished, False)

Parquet files without schema

In [2]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T

def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",     # "date" or "timestamp"
    ingest_date: Optional[str] = None  # e.g., "2026-02-03" (casts to date); if None -> current_date()
) -> DataFrame:
    """
    1) Standardizes the provided or auto-detected date/timestamp columns:
       - Parses common string formats to timestamp
       - Casts to `date` (default) or keeps as `timestamp` via `cast_dates_as`
    2) Adds `ingest_date` column (date) for downstream partitioning.

    Parameters
    ----------
    df : input DataFrame
    date_cols : list of column names to treat as dates/timestamps. If None, auto-detects by name/ctype.
    input_formats : list of strptime-style formats to try in order
    cast_dates_as : "date" or "timestamp"
    ingest_date : string literal yyyy-MM-dd; if None uses current_date()
    """

    # --- 0) Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # --- 1) Which columns are date-like?
    if date_cols is None:
        # Heuristic: names that look like dates & are string/date/timestamp
        keywords = ("date", "dt", "_at", "_on", "timestamp", "ts", "fecha", "created", "updated")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp") and any(k in c.lower() for k in keywords)
        ]

    # --- 2) Parse formats for string dates
    if input_formats is None:
        input_formats = [
            # Date only
            "yyyy-MM-dd", "yyyy/MM/dd", "dd-MM-yyyy", "dd/MM/yyyy", "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",
            # Date + time
            "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss",
            # ISO-ish / with millis / TZ
            "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssXXX", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)
    for c in date_cols:
        t = dtype_map.get(c, "")

        # If it's already timestamp/date, just unify type later
        if t == "string":
            # Try multiple formats with coalesce
            parsed = None
            for fmt in input_formats:
                candidate = F.to_timestamp(F.col(c), fmt)
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # Epoch seconds / millis (string or numeric) fallbacks
            parsed = F.coalesce(
                parsed,
                F.to_timestamp(F.col(c).cast("double")),                    # epoch seconds
                F.to_timestamp((F.col(c).cast("double")/1000.0))            # epoch millis
            )

            # If nothing matched, keep original to avoid data loss
            df = df.withColumn(c, F.when(parsed.isNotNull(), parsed).otherwise(F.col(c)))

        # Finally, cast to the requested final type
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # --- 3) Add ingest_date for partitioning
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df

StatementMeta(, 42b8ced0-3399-483a-977c-e778d114b0d4, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, DecimalType, TimestampType

# -------------------------------------------------------------
# 0) Paths (ajústalos a tu storage)
# -------------------------------------------------------------
bronze_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Production.Product.parquet/"
base_silver_root = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"
schema_name = "Production"
table_name  = "Product"


base_folder   = f"{(schema_name)}_{(table_name)}"      # -> production_product
silver_folder = f"silver_{base_folder}"                                # -> silver_production_product
silver_path   = f"{base_silver_root}{silver_folder}/"
#display(silver_path)

# -------------------------------------------------------------
# 1) Tu SCHEMA (ya lo tienes definido como schemaProduct)
#    *Asegúrate de que esté en el notebook antes de este bloque*
# -------------------------------------------------------------
schemaProduct = StructType([
    StructField("ProductID",               IntegerType(),    True),
    StructField("Name",                    StringType(),     True),
    StructField("ProductNumber",           StringType(),     True),
    StructField("MakeFlag",                BooleanType(),    True),
    StructField("FinishedGoodsFlag",       BooleanType(),    True),
    StructField("Color",                   StringType(),     True),
    StructField("SafetyStockLevel",        IntegerType(),    True),
    StructField("ReorderPoint",            IntegerType(),    True),
    StructField("StandardCost",            DecimalType(19,4),True),
    StructField("ListPrice",               DecimalType(19,4),True),
    StructField("Size",                    StringType(),     True),
    StructField("SizeUnitMeasureCode",     StringType(),     True),
    StructField("WeightUnitMeasureCode",   StringType(),     True),
    StructField("Weight",                  DecimalType(8,2), True),
    StructField("DaysToManufacture",       IntegerType(),    True),
    StructField("ProductLine",             StringType(),     True),
    StructField("Class",                   StringType(),     True),
    StructField("Style",                   StringType(),     True),
    StructField("ProductSubcategoryID",    IntegerType(),    True),
    StructField("ProductModelID",          IntegerType(),    True),
    StructField("SellStartDate",           TimestampType(),  True),
    StructField("SellEndDate",             TimestampType(),  True),
    StructField("DiscontinuedDate",        TimestampType(),  True),
    StructField("rowguid",                 StringType(),     True),
    StructField("ModifiedDate",            TimestampType(),  True)
])

# Orden definitivo de columnas según tu schema
final_cols = [f.name for f in schemaProduct]

# -------------------------------------------------------------
# 2) Leer el parquet de bronze
# -------------------------------------------------------------
df_bronze = spark.read.parquet(bronze_path)

# -------------------------------------------------------------
# 3) Alinear nombres de columnas
#    - Si el parquet ya tiene la misma cantidad de columnas pero con
#      nombres distintos/abrev., las renombramos siguiendo schemaProduct.
#    - Si faltan columnas, las creamos como null (cast al tipo correcto).
# -------------------------------------------------------------
df = df_bronze

existing_cols = df.columns

# 3a) Si el número de columnas coincide, renombra por posición (seguro cuando viene de ORC/Parquet sin headers "correctos")
if len(existing_cols) == len(final_cols):
    for old_name, new_name in zip(existing_cols, final_cols):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

# 3b) Si faltan columnas según el schema, créalas con null
for f in final_cols:
    if f not in df.columns:
        df = df.withColumn(f, lit(None).cast([sf.dataType for sf in schemaProduct if sf.name == f][0]))

# -------------------------------------------------------------
# 4) Castear tipos exactamente como en schemaProduct
# -------------------------------------------------------------
for sf in schemaProduct:
    # Sólo castear si la columna existe (ya debería existir por los pasos previos)
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

# -------------------------------------------------------------
# 5) Reordenar columnas y escribir a silver en Parquet
#    - Snappy por defecto; puedes cambiar compresión si quieres
#    - Quita .coalesce(1) si no necesitas un único archivo
# -------------------------------------------------------------
df_final = df.select([col(c) for c in final_cols])

#clean
df_clean = clean_df(df_final)
# dates
df_table= standardizedate_df(df_clean)
#display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)   


StatementMeta(, 75631c45-4443-43a2-96c8-37202a411354, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd454995-f36e-4964-ae1a-2cc2cc4f111e)

In [5]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
bronze_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Production.ProductModelProductDescriptionCulture.parquet/"
base_silver_root = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

schema_name = "Production"
table_name  = "ProductModelProductDescriptionCulture"

# Carpeta final con prefijo silver_
base_folder   = f"{schema_name}_{table_name}"        # Production_ProductModelProductDescriptionCulture
silver_folder = f"silver_{base_folder}".lower()      # silver_production_productmodelproductdescriptionculture
silver_path   = f"{base_silver_root}{silver_folder}/"

#display(silver_path)


schemaPMDC = StructType([
    StructField("ProductModelID",        IntegerType(),   True),
    StructField("ProductDescriptionID",  IntegerType(),   True),
    StructField("CultureID",             StringType(),    True),   # nchar(6) → String
    StructField("ModifiedDate",          TimestampType(), True)
])

final_cols = [f.name for f in schemaPMDC]

df_bronze = spark.read.parquet(bronze_path)
df = df_bronze
existing_cols = df.columns

# 4a) Si coinciden en cantidad, renombrar por posición
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b) Si falta alguna columna del schema → crearla como null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaPMDC if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaPMDC:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

# Limpiezas (tu funciones)
df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

df_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)


# Register Delta table in the Lakehouse metastore
spark.sql(f"""
CREATE OR REPLACE TABLE silver.{schema_name}_{table_name}
USING DELTA
LOCATION '{silver_path}'
""")
#print("Silver guardado en:", silver_path)

StatementMeta(, 75631c45-4443-43a2-96c8-37202a411354, 7, Finished, Available, Finished, False)

'abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productmodelproductdescriptionculture/'

SynapseWidget(Synapse.DataFrame, 6623d7a6-0fee-4493-a5c9-1027496c2727)

Silver guardado en: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productmodelproductdescriptionculture/


In [6]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
bronze_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Production.ProductDocument.parquet/"
base_silver_root = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

schema_name = "Production"
table_name  = "ProductDocument"

# Carpeta silver
base_folder   = f"{schema_name}_{table_name}"
silver_folder = f"silver_{base_folder}".lower()
silver_path   = f"{base_silver_root}{silver_folder}/"

#display(silver_path)

#schema
schemaProductDocument = StructType([
    StructField("ProductID",     IntegerType(),   True),
    StructField("DocumentNode",  StringType(),    True),   # hierarchyid -> string
    StructField("ModifiedDate",  TimestampType(), True)
])

final_cols = [f.name for f in schemaProductDocument]

#read parquet
df_bronze = spark.read.parquet(bronze_path)
df = df_bronze
existing_cols = df.columns

# 4a. Si coincide el número de columnas, renombra por posición
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b. Crear columnas faltantes como null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaProductDocument if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaProductDocument:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))


df_final = df.select([col(c) for c in final_cols])

#clean
df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

df_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

#print("Silver guardado en:", silver_path)

StatementMeta(, 75631c45-4443-43a2-96c8-37202a411354, 8, Finished, Available, Finished, False)

'abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productdocument/'

SynapseWidget(Synapse.DataFrame, 48401211-2f51-4cb6-ad82-7c67b1c873f2)

Silver guardado en: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productdocument/


In [7]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, BinaryType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
bronze_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Production.ProductPhoto.parquet/"
base_silver_root = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

schema_name = "Production"
table_name  = "ProductPhoto"

# Carpeta silver con prefijo:
base_folder   = f"{schema_name}_{table_name}"
silver_folder = f"silver_{base_folder}".lower()
silver_path   = f"{base_silver_root}{silver_folder}/"

#display(silver_path)

#schema
schemaProductPhoto = StructType([
    StructField("ProductPhotoID",          IntegerType(),   True),
    StructField("ThumbNailPhoto",          BinaryType(),    True),   # varbinary(max) -> BinaryType
    StructField("ThumbnailPhotoFileName",  StringType(),    True),
    StructField("LargePhoto",              BinaryType(),    True),   # varbinary(max) -> BinaryType
    StructField("LargePhotoFileName",      StringType(),    True),
    StructField("ModifiedDate",            TimestampType(), True)
])

final_cols = [f.name for f in schemaProductPhoto]

#Read parquet
df_bronze = spark.read.parquet(bronze_path)
df = df_bronze
existing_cols = df.columns


# 4a — Renombrar si coincide la cantidad de columnas
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b — Crear columnas faltantes como null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaProductPhoto if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))


for sf in schemaProductPhoto:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

#save
df_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

#print("Silver guardado en:", silver_path)

StatementMeta(, 75631c45-4443-43a2-96c8-37202a411354, 9, Finished, Available, Finished, False)

'abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productphoto/'

SynapseWidget(Synapse.DataFrame, 812a1c91-316e-4728-a7b2-d1c6b3ecd689)

Silver guardado en: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_productphoto/


In [8]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, TimestampType, BinaryType
)

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
bronze_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Production.Document.parquet/"
base_silver_root = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

schema_name = "Production"
table_name  = "Document"

# Carpeta final
base_folder   = f"{schema_name}_{table_name}"
silver_folder = f"silver_{base_folder}".lower()
silver_path   = f"{base_silver_root}{silver_folder}/"

#display(silver_path)

#schema
schemaDocument = StructType([
    StructField("DocumentNode",            StringType(),    True),   # hierarchyid -> string
    # DocumentLevel SE IGNORA → columna calculada
    StructField("Title",                   StringType(),    True),
    StructField("Owner",                   IntegerType(),   True),
    StructField("FolderFlag",              IntegerType(),   True),   # bit -> tinyint -> integer
    StructField("FileName",                StringType(),    True),
    StructField("FileExtension",           StringType(),    True),
    StructField("Revision",                StringType(),    True),
    StructField("ChangeNumber",            IntegerType(),   True),
    StructField("Status",                  IntegerType(),   True),   # tinyint -> integer
    StructField("DocumentSummary",         StringType(),    True),   # nvarchar(max)
    StructField("Document",                BinaryType(),    True),   # varbinary(max)
    StructField("rowguid",                 StringType(),    True),   # uniqueidentifier → string
    StructField("ModifiedDate",            TimestampType(), True)
])

final_cols = [f.name for f in schemaDocument]

df_bronze = spark.read.parquet(bronze_path)
df = df_bronze
existing_cols = df.columns

# 4a — Renombrar por posición si tienen igual número de columnas
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b — Crear las columnas faltantes
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaDocument if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaDocument:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

#save in delta
df_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

#print("Silver guardado en:", silver_path)

StatementMeta(, 75631c45-4443-43a2-96c8-37202a411354, 10, Finished, Available, Finished, False)

'abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_document/'

SynapseWidget(Synapse.DataFrame, 9c69d944-4925-4a58-abf6-8c00249e21ad)

Silver guardado en: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_document/
